# P0 — Playground: tiny graphs

A hands-on warm-up. We draw small graphs and count their **paths**. The whole project rests on one fact:
*how many paths connect two nodes* decides how much information gets squashed together. Edit the edge lists
and re-run — everything updates.

**You will see 3+ figures:** a line graph, a two-route graph, and a diamond (plus a double-diamond).

In [ ]:
import numpy as np, torch
from oversquash import viz
from oversquash.ideal_bridge import walk_operators

def path_counts(edges, n, src, dst, max_len=4):
    raw, eff = walk_operators(np.array(edges).T, n, max_length=max_len)
    for g in range(max_len):
        r, e = int(raw[g][src, dst]), int(eff[g][src, dst])
        if r:
            print(f'  length {g+1}: {r} path(s)' + ('' if r == e else f'  (kQ/I de-duplicates -> {e})'))

## Figure 1 — a straight line (no redundancy)

`0→1→2→3→4`: exactly **one** path end to end. Far apart, but no pile-up — so a normal GNN can (slowly) carry
the signal. Over-squashing needs *many* competing paths, which a line doesn't have.

In [ ]:
line = [(0,1),(1,2),(2,3),(3,4)]
viz.draw_graph(line, 5, src=0, dst=4, pos={i:(i,0) for i in range(5)}, title='line: one path 0->4')
path_counts(line, 5, 0, 4)

## Figure 2 — two routes of different length

A short edge `0→3` and a long detour `0→1→2→3`. Information reaches `3` at **two different ranges** (length 1
and length 3). This is why we index everything by the hop-count `g`.

In [ ]:
two = [(0,3),(0,1),(1,2),(2,3)]
viz.draw_graph(two, 4, src=0, dst=3, pos={0:(0,0),1:(1,1),2:(2,1),3:(3,0)}, title='two routes 0->3')
path_counts(two, 4, 0, 3)

## Figure 3 — parallel routes, same length (redundancy)

The 'diamond': `0→1→3` and `0→2→3`, both length 2. Two paths, but if the middle nodes carry the *same*
content they are **redundant** — watch `kQ/I` collapse the count from 2 to 1.

In [ ]:
diamond = [(0,1),(0,2),(1,3),(2,3)]
viz.draw_graph(diamond, 4, src=0, dst=3, pos={0:(0,0),1:(1,1),2:(1,-1),3:(2,0)}, title='diamond: 2 parallel paths')
path_counts(diamond, 4, 0, 3)

## Figure 4 — your turn: a double diamond

Two diamonds in a row give **4** parallel paths of length 4. Edit `my_graph` and re-run to explore.

In [ ]:
my_graph = [(0,1),(0,2),(1,3),(2,3),(3,4),(3,5),(4,6),(5,6)]
viz.draw_graph(my_graph, 7, src=0, dst=6, title='double diamond: 4 parallel paths 0->6')
path_counts(my_graph, 7, 0, 6, max_len=4)
print('\nMore parallel same-length paths = worse over-squashing. On to P1!')